## Dependencies



In [1]:
!pip install  transformers
!pip install  accelerate
!pip install -U datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.4/302.4 kB 3.0 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-many

In [2]:
import pandas as pd
from datasets import load_dataset

In [3]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
from datasets import DatasetDict, Dataset
data = DatasetDict.load_from_disk('/content/drive/MyDrive/complete_dataset')

In [8]:


tag2index = {"O": 0, "B-Task": 1, "I-Task": 2, "B-Process": 3, "I-Process": 4, "B-Material": 5, "I-Material": 6}
index2tag = {0:"O", 1: "B-Task", 2: "I-Task", 3: "B-Process", 4: "I-Process", 5: "B-Material", 6: "I-Material"}


In [ ]:
def create_tag_names(batch):
  tag_name = {'ner_tags_str': [index2tag[idx] for idx in batch['tags_idx']]}
  return tag_name
data = data.map(create_tag_names)

## Model Building

### Tokenization

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [11]:
inputs = data['train'][0]['tokens']
inputs = tokenizer(inputs, is_split_into_words=True)
print(inputs.tokens())

['[CLS]', 'With', 'employment', 'of', 'utilizing', 'the', 'investigation', ',', 'expert', 'interviews', 'and', 'comparison', ',', 'this', 'article', 'investigate', '##s', 'the', 'cu', '##rri', '##cula', 'construction', ',', 'cu', '##rri', '##cula', 'design', 'and', 'cu', '##rri', '##cula', 'content', 'for', 'sports', 'free', 'normal', 'students', '.', 'On', 'the', 'basis', 'of', 'the', 'investigation', ',', 'this', 'article', 'analyzed', 'the', 'theoretical', 'framework', 'of', 'cu', '##rricular', 'construction', 'and', 'proposed', 'some', 'suggestions', '.', 'We', 'hope', 'that', 'it', 'can', 'provide', 'some', 'evidence', '##s', 'for', 'cu', '##rri', '##cula', 'design', 'for', 'sports', 'free', 'normal', 'students', '.', '[SEP]']


In [12]:
print(data['train'][0]['tokens'])
print(data['train'][0]['str_tag'])


['With', 'employment', 'of', 'utilizing', 'the', 'investigation,', 'expert', 'interviews', 'and', 'comparison,', 'this', 'article', 'investigates', 'the', 'curricula', 'construction,', 'curricula', 'design', 'and', 'curricula', 'content', 'for', 'sports', 'free', 'normal', 'students.', 'On', 'the', 'basis', 'of', 'the', 'investigation,', 'this', 'article', 'analyzed', 'the', 'theoretical', 'framework', 'of', 'curricular', 'construction', 'and', 'proposed', 'some', 'suggestions.', 'We', 'hope', 'that', 'it', 'can', 'provide', 'some', 'evidences', 'for', 'curricula', 'design', 'for', 'sports', 'free', 'normal', 'students.']
['O', 'O', 'O', 'O', 'B-Process', 'B-Process', 'I-Process', 'O', 'O', 'O', 'O', 'B-Task', 'I-Task', 'I-Task', 'I-Task', 'I-Task', 'I-Task', 'I-Task', 'I-Task', 'I-Task', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Task', 'O', 'O', 'O', 'O', 'B-Process', 'I-Process', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Process', '

In [13]:
def align_labels_with_tokens(labels, word_ids):
  new_labels = []
  current_word=None
  for word_id in word_ids:
    if word_id != current_word:
      current_word = word_id
      label = -100 if word_id is None else labels[word_id]
      new_labels.append(label)

    elif word_id is None:
      new_labels.append(-100)

    else:
      label = labels[word_id]

      if label%2==1:
        label = label + 1
      new_labels.append(label)

  return new_labels


In [ ]:
labels = data['train'][0]['tags_idx']
word_ids = inputs.word_ids()
align_labels_with_tokens(labels, word_ids)
labels

In [15]:
def tokenize_and_align_labels(examples):
  tokenized_inputs = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)

  all_labels = examples['tags_idx']

  new_labels = []
  for i, labels in enumerate(all_labels):
    word_ids = tokenized_inputs.word_ids(i)
    new_labels.append(align_labels_with_tokens(labels, word_ids))

  tokenized_inputs['labels'] = new_labels

  return tokenized_inputs


In [ ]:
tokenized_datasets = data.map(tokenize_and_align_labels, batched=True, remove_columns=data['train'].column_names)

### Data Collation and Metrics

In [17]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [18]:
batch = data_collator([tokenized_datasets['train'][i] for i in range(2)])
# batch

### Metrics

In [ ]:
!pip install seqeval
!pip install evaluate

import evaluate
metric = evaluate.load('seqeval')

In [20]:
label_names = list(tag2index.keys())
labels = data['train'][0]['tags_idx']
labels = [label_names[i] for i in labels]
# labels

In [21]:
import numpy as np

def compute_metrics(eval_preds):
  logits, labels = eval_preds

  predictions = np.argmax(logits, axis=-1)

  true_labels = [[label_names[l] for l in label if l!=-100] for label in labels]

  true_predictions = [[label_names[p] for p,l in zip(prediction, label) if l!=-100]
                      for prediction, label in zip(predictions, labels)]

  all_metrics = metric.compute(predictions=true_predictions, references=true_labels)

  return {"precision": all_metrics['overall_precision'],
          "recall": all_metrics['overall_recall'],
          "f1": all_metrics['overall_f1'],
          "accuracy": all_metrics['overall_accuracy']}

### Model Training

In [22]:
id2label = {i:label for i, label in enumerate(label_names)}
label2id = {label:i for i, label in enumerate(label_names)}

In [23]:
print(id2label)

{0: 'O', 1: 'B-Task', 2: 'I-Task', 3: 'B-Process', 4: 'I-Process', 5: 'B-Material', 6: 'I-Material'}


In [44]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
                                                    model_checkpoint,
                                                    id2label=id2label,
                                                    label2id=label2id)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [45]:
from transformers import TrainingArguments

args = TrainingArguments("distilbert-finetuned-ner",
                         evaluation_strategy = "epoch",
                         save_strategy="epoch",
                         learning_rate = 2e-5,
                         num_train_epochs=9,
                         weight_decay=0.01)

In [ ]:
from transformers import Trainer
trainer = Trainer(model=model,
                  args=args,
                  train_dataset = tokenized_datasets['train'],
                  eval_dataset = tokenized_datasets['test'],
                  data_collator=data_collator,
                  compute_metrics=compute_metrics,
                  tokenizer=tokenizer)

trainer.train()

In [48]:
test_results = trainer.evaluate(eval_dataset=tokenized_datasets['test'])
print(f"Test results: {test_results}")

Test results: {'eval_loss': 0.5736083388328552, 'eval_precision': 0.2928930366116296, 'eval_recall': 0.35758106923751093, 'eval_f1': 0.3220205209155485, 'eval_accuracy': 0.8012604741101482, 'eval_runtime': 0.9949, 'eval_samples_per_second': 50.256, 'eval_steps_per_second': 7.036, 'epoch': 9.0}


In [49]:
test_results = trainer.evaluate(eval_dataset=tokenized_datasets['validation'])
print(f"Validation results: {test_results}")

Validation results: {'eval_loss': 0.4777069091796875, 'eval_precision': 0.3386243386243386, 'eval_recall': 0.4052464947987336, 'eval_f1': 0.36895202800082355, 'eval_accuracy': 0.8432310548018587, 'eval_runtime': 7.5137, 'eval_samples_per_second': 46.449, 'eval_steps_per_second': 5.856, 'epoch': 9.0}
